In [16]:
import pandas as pd 
import calendar as cal 
import plotly.express as px
df = pd.read_csv("data.csv")


# 2. Garder uniquement les colonnes utiles
cols = [
    'CustomerID',
    'Gender',
    'Location',
    'Product_Category',
    'Quantity',
    'Avg_Price',
    'Transaction_Date',
    'Month',
    'Discount_pct'
]

df = df[cols]

# 3. Remplacer les valeurs manquantes dans CustomerID par 0 et convertir en entier
df['CustomerID'] = df['CustomerID'].fillna(0).astype(int)

# 4. Convertir Transaction_Date en date
df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'])

# 5. Créer Total_price avec la remise
df['Total_price'] = df['Quantity'] * df['Avg_Price'] * (1 - df['Discount_pct'] / 100)

# 6. Calculer le chiffre d'affaires total
def chiffre_affaire(data):
    chiffre_affaire = data['Total_price'].sum()
    return chiffre_affaire

print(chiffre_affaire(df))

# 7. Fréquence des meilleures ventes

def meilleure_vente(data, top=10, ascending=False):
    freq = data.groupby("Product_Category")["Quantity"].sum().sort_values(ascending=ascending)
    return freq.head(top)

top=meilleure_vente(df, top=10, ascending= False).reset_index()

print("Le top 10 des meilleurs ventes est :",top )

def indicateur_du_mois(data, current_month=12, freq=True, abbr=False):
    data_mois = data[data["Month"] == current_month]
    if freq:
        result = data_mois["Quantity"].sum()
    else:
        result = data_mois["Total_price"].sum()
    if abbr:
        month_name = cal.month_abbr[current_month]
    else:
        month_name = cal.month_name[current_month]
    return month_name, result

print(indicateur_du_mois(df , 12 , True , False))



3717667.247
Le top 10 des meilleurs ventes est :        Product_Category  Quantity
0                Office   88383.0
1               Apparel   32438.0
2             Drinkware   30501.0
3             Lifestyle   24881.0
4              Nest-USA   21430.0
5                  Bags   15273.0
6  Notebooks & Journals    9556.0
7              Headgear    3533.0
8                  Nest    2837.0
9            Housewares    2484.0
('December', np.float64(12667.0))


In [18]:
#Barplot

def barplot_top_10_ventes(data):

    # récupérer le top 10
    top = meilleure_vente(data, top=10, ascending=False).reset_index()

    # filtrer les données pour ces catégories
    data_top = data[data["Product_Category"].isin(top["Product_Category"])]

    # ventes par catégorie et genre
    ventes = data_top.groupby(["Product_Category", "Gender"])["Quantity"].sum().reset_index()

    fig = px.bar(
        ventes,
        x="Quantity",
        y="Product_Category",
        color="Gender",
        barmode="group",
        title="Top 10 des meilleures ventes par genre",
        labels={
            "Product_Category": "Catégorie de produits",
            "Quantity": "Quantité vendue",
            "Gender": "Genre"
        },
        category_orders={
            "Product_Category": top["Product_Category"].tolist()
        }
    )
    return fig
barplot_top_10_ventes(df)

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'alignmentgroup': 'True',
              'hovertemplate': 'Genre=F<br>Quantité vendue=%{x}<br>Catégorie de produits=%{y}<extra></extra>',
              'legendgroup': 'F',
              'marker': {'color': '#636efa', 'pattern': {'shape': ''}},
              'name': 'F',
              'offsetgroup': 'F',
              'orientation': 'h',
              'showlegend': True,
              'textposition': 'auto',
              'type': 'bar',
              'x': {'bdata': ('AAAAAIBQ1EAAAAAAgEjFQAAAAACAvt' ... 'AAgPnJQAAAAAAAG7NAAAAAAGBz7EA='),
                    'dtype': 'f8'},
              'xaxis': 'x',
              'y': array(['Apparel', 'Bags', 'Drinkware', 'Headgear', 'Housewares', 'Lifestyle',
                          'Nest', 'Nest-USA', 'Notebooks & Journals', 'Office'], dtype=object),
              'yaxis': 'y'},
             {'alignmentgroup': 'True',
              'hovertemplate': 'Genre=M<br>Quantité vendue=%{x}<br>Catégorie de produits=%{y}<extra></extra>',
              'legendgroup': 'M',
              'marker': {'color': '#EF553B', 'pattern': {'shape': ''}},
              'name': 'M',
              'offsetgroup': 'M',
              'orientation': 'h',
              'showlegend': True,
              'textposition': 'auto',
              'type': 'bar',
              'x': {'bdata': ('AAAAAAC6xkAAAAAAABixQAAAAACAFc' ... 'AAAMO/QAAAAAAAObJAAAAAAABp3UA='),
                    'dtype': 'f8'},
              'xaxis': 'x',
              'y': array(['Apparel', 'Bags', 'Drinkware', 'Headgear', 'Housewares', 'Lifestyle',
                          'Nest', 'Nest-USA', 'Notebooks & Journals', 'Office'], dtype=object),
              'yaxis': 'y'}],
    'layout': {'barmode': 'group',
               'legend': {'title': {'text': 'Genre'}, 'tracegroupgap': 0},
               'template': '...',
               'title': {'text': 'Top 10 des meilleures ventes par genre'},
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': 'Quantité vendue'}},
               'yaxis': {'anchor': 'x',
                         'categoryarray': [Housewares, Nest, Headgear, Notebooks &
                                           Journals, Bags, Nest-USA, Lifestyle,
                                           Drinkware, Apparel, Office],
                         'categoryorder': 'array',
                         'domain': [0.0, 1.0],
                         'title': {'text': 'Catégorie de produits'}}}
})

In [20]:
def plot_evolution_chiffre_affaire(data):

    # mettre la date comme index
    data = data.set_index("Transaction_Date")

    # chiffre d'affaire par semaine
    evolution = data["Total_price"].resample("W").sum().reset_index() #W = Week

    # graphique
    fig = px.line(
        evolution,
        x="Transaction_Date",
        y="Total_price",
        title="Evolution du chiffre d'affaire par semaine",
        labels={
        "Transaction_Date": "Semaine",
        "Total_price": "Chiffre d'affaire"
        }
    )
    fig.update_layout(
    margin=dict(l=10, r=10, t=30, b=10)
)

    return fig
plot_evolution_chiffre_affaire(df)

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'hovertemplate': "Semaine=%{x}<br>Chiffre d'affaire=%{y}<extra></extra>",
              'legendgroup': '',
              'line': {'color': '#636efa', 'dash': 'solid'},
              'marker': {'symbol': 'circle'},
              'mode': 'lines',
              'name': '',
              'orientation': 'v',
              'showlegend': False,
              'type': 'scatter',
              'x': array(['2019-01-06T00:00:00.000000000', '2019-01-13T00:00:00.000000000',
                          '2019-01-20T00:00:00.000000000', '2019-01-27T00:00:00.000000000',
                          '2019-02-03T00:00:00.000000000', '2019-02-10T00:00:00.000000000',
                          '2019-02-17T00:00:00.000000000', '2019-02-24T00:00:00.000000000',
                          '2019-03-03T00:00:00.000000000', '2019-03-10T00:00:00.000000000',
                          '2019-03-17T00:00:00.000000000', '2019-03-24T00:00:00.000000000',
                          '2019-03-31T00:00:00.000000000', '2019-04-07T00:00:00.000000000',
                          '2019-04-14T00:00:00.000000000', '2019-04-21T00:00:00.000000000',
                          '2019-04-28T00:00:00.000000000', '2019-05-05T00:00:00.000000000',
                          '2019-05-12T00:00:00.000000000', '2019-05-19T00:00:00.000000000',
                          '2019-05-26T00:00:00.000000000', '2019-06-02T00:00:00.000000000',
                          '2019-06-09T00:00:00.000000000', '2019-06-16T00:00:00.000000000',
                          '2019-06-23T00:00:00.000000000', '2019-06-30T00:00:00.000000000',
                          '2019-07-07T00:00:00.000000000', '2019-07-14T00:00:00.000000000',
                          '2019-07-21T00:00:00.000000000', '2019-07-28T00:00:00.000000000',
                          '2019-08-04T00:00:00.000000000', '2019-08-11T00:00:00.000000000',
                          '2019-08-18T00:00:00.000000000', '2019-08-25T00:00:00.000000000',
                          '2019-09-01T00:00:00.000000000', '2019-09-08T00:00:00.000000000',
                          '2019-09-15T00:00:00.000000000', '2019-09-22T00:00:00.000000000',
                          '2019-09-29T00:00:00.000000000', '2019-10-06T00:00:00.000000000',
                          '2019-10-13T00:00:00.000000000', '2019-10-20T00:00:00.000000000',
                          '2019-10-27T00:00:00.000000000', '2019-11-03T00:00:00.000000000',
                          '2019-11-10T00:00:00.000000000', '2019-11-17T00:00:00.000000000',
                          '2019-11-24T00:00:00.000000000', '2019-12-01T00:00:00.000000000',
                          '2019-12-08T00:00:00.000000000', '2019-12-15T00:00:00.000000000',
                          '2019-12-22T00:00:00.000000000', '2019-12-29T00:00:00.000000000',
                          '2020-01-05T00:00:00.000000000'], dtype='datetime64[ns]'),
              'xaxis': 'x',
              'y': {'bdata': ('GQRWDjVU9EDZzvdTsVzyQAIrhxY5jv' ... 'xxYPdA0k1iEAgX6EDx0k1i4OfIQA=='),
                    'dtype': 'f8'},
              'yaxis': 'y'}],
    'layout': {'legend': {'tracegroupgap': 0},
               'margin': {'b': 10, 'l': 10, 'r': 10, 't': 30},
               'template': '...',
               'title': {'text': "Evolution du chiffre d'affaire par semaine"},
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': 'Semaine'}},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'title': {'text': "Chiffre d'affaire"}}}
})

In [34]:
#3. `plot_chiffre_affaire_mois(data)`
import plotly.graph_objects as go 
def plot_chiffre_affaire_mois(data, current_month):
    mois_courant = indicateur_du_mois(data, current_month=current_month, freq=False, abbr=False)
    mois_precedent = indicateur_du_mois(data, current_month=current_month-1, freq=False, abbr=False)

    fig = go.Figure()

    fig.add_trace(
        go.Indicator(
            value=mois_courant[1],
            mode="number+delta",
            delta={"reference": mois_precedent[1]},
            title={"text": f"{mois_courant[0]}"}
        )
    )

    fig.update_layout(
        margin=dict(l=10, r=10, t=30, b=10)
    )
    return fig
plot_chiffre_affaire_mois(df, 12)

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'delta': {'reference': np.float64(406866.128)},
              'mode': 'number+delta',
              'title': {'text': 'December'},
              'type': 'indicator',
              'value': np.float64(366280.73299999995)}],
    'layout': {'margin': {'b': 10, 'l': 10, 'r': 10, 't': 30}, 'template': '...'}
})

In [33]:
import plotly.graph_objects as go

def plot_vente_mois(data, current_month, abbr=False):
    mois_courant = indicateur_du_mois(data, current_month=current_month, freq=True, abbr=abbr)
    mois_precedent = indicateur_du_mois(data, current_month=current_month-1, freq=True, abbr=abbr)

    fig = go.Figure()

    fig.add_trace(
        go.Indicator(
            mode="number+delta",
            value=mois_courant[1],
            delta={"reference": mois_precedent[1]},
            title={"text": f"{mois_courant[0]}"}
        )
    )

    fig.update_layout(
        margin=dict(l=10, r=10, t=30, b=10)
    )

    return fig
plot_vente_mois(df,12)

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'delta': {'reference': np.float64(15120.0)},
              'mode': 'number+delta',
              'title': {'text': 'December'},
              'type': 'indicator',
              'value': np.float64(12667.0)}],
    'layout': {'margin': {'b': 10, 'l': 10, 'r': 10, 't': 30}, 'template': '...'}
})

In [36]:
#5. Table des 100 dernières ventes

df_last_100 = df.sort_values("Transaction_Date", ascending=False).head(100)

df_last_100["Transaction_Date"] = pd.to_datetime(df_last_100["Transaction_Date"]).dt.strftime("%Y-%m-%d")

df_last_100 = df_last_100[
    ["Transaction_Date", "Gender", "Location", "Product_Category",
     "Quantity", "Avg_Price", "Discount_pct"]]

In [ ]:
import pandas as pd 
import calendar as cal 
import plotly.express as px
df = pd.read_csv("data.csv")


# 2. Garder uniquement les colonnes utiles
cols = [
    'CustomerID',
    'Gender',
    'Location',
    'Product_Category',
    'Quantity',
    'Avg_Price',
    'Transaction_Date',
    'Month',
    'Discount_pct'
]

df = df[cols]

# 3. Remplacer les valeurs manquantes dans CustomerID par 0 et convertir en entier
df['CustomerID'] = df['CustomerID'].fillna(0).astype(int)

# 4. Convertir Transaction_Date en date
df['Transaction_Date'] = pd.to_datetime(df['Transaction_Date'])

# 5. Créer Total_price avec la remise
df['Total_price'] = df['Quantity'] * df['Avg_Price'] * (1 - df['Discount_pct'] / 100)

# 6. Calculer le chiffre d'affaires total
def chiffre_affaire(data):
    chiffre_affaire = data['Total_price'].sum()
    return chiffre_affaire


# 7. Fréquence des meilleures ventes

def meilleure_vente(data, top=10, ascending=False):
    freq = data.groupby("Product_Category")["Quantity"].sum().sort_values(ascending=ascending)
    return freq.head(top)

top=meilleure_vente(df, top=10, ascending= False).reset_index()


def indicateur_du_mois(data, current_month=12, freq=True, abbr=False):
    data_mois = data[data["Month"] == current_month]
    if freq:
        result = data_mois["Quantity"].sum()
    else:
        result = data_mois["Total_price"].sum()
    if abbr:
        month_name = cal.month_abbr[current_month]
    else:
        month_name = cal.month_name[current_month]
    return month_name, result


#Barplot

def barplot_top_10_ventes(data):

    # récupérer le top 10
    top = meilleure_vente(data, top=10, ascending=False).reset_index()

    # filtrer les données pour ces catégories
    data_top = data[data["Product_Category"].isin(top["Product_Category"])]

    # ventes par catégorie et genre
    ventes = data_top.groupby(["Product_Category", "Gender"])["Quantity"].sum().reset_index()

    fig = px.bar(
        ventes,
        x="Quantity",
        y="Product_Category",
        color="Gender",
        barmode="group",
        title="Top 10 des meilleures ventes par genre",
        labels={
            "Product_Category": "Catégorie de produits",
            "Quantity": "Quantité vendue",
            "Gender": "Genre"
        },
        category_orders={
            "Product_Category": top["Product_Category"].tolist()
        }
    )
    return fig

def plot_evolution_chiffre_affaire(data):

    # mettre la date comme index
    data = data.set_index("Transaction_Date")

    # chiffre d'affaire par semaine
    evolution = data["Total_price"].resample("W").sum().reset_index() #W = Week

    # graphique
    fig = px.line(
        evolution,
        x="Transaction_Date",
        y="Total_price",
        title="Evolution du chiffre d'affaire par semaine",
        labels={
        "Transaction_Date": "Semaine",
        "Total_price": "Chiffre d'affaire"
        }
    )
    fig.update_layout(
    margin=dict(l=10, r=10, t=30, b=10)
)

    return fig


#3. `plot_chiffre_affaire_mois(data)`
import plotly.graph_objects as go 
def plot_chiffre_affaire_mois(data, current_month):
    mois_courant = indicateur_du_mois(data, current_month=current_month, freq=False, abbr=False)
    mois_precedent = indicateur_du_mois(data, current_month=current_month-1, freq=False, abbr=False)

    fig = go.Figure()

    fig.add_trace(
        go.Indicator(
            value=mois_courant[1],
            mode="number+delta",
            delta={"reference": mois_precedent[1]},
            title={"text": f"{mois_courant[0]}"}
        )
    )

    fig.update_layout(
        margin=dict(l=10, r=10, t=30, b=10)
    )
    return fig

#Quantités vendues
def plot_vente_mois(data, current_month, abbr=False):
    mois_courant = indicateur_du_mois(data, current_month=current_month, freq=True, abbr=abbr)
    mois_precedent = indicateur_du_mois(data, current_month=current_month-1, freq=True, abbr=abbr)

    fig = go.Figure()

    fig.add_trace(
        go.Indicator(
            mode="number+delta",
            value=mois_courant[1],
            delta={"reference": mois_precedent[1]},
            title={"text": f"{mois_courant[0]}"}
        )
    )

    fig.update_layout(
        margin=dict(l=10, r=10, t=30, b=10)
    )

    return fig


#5. Table des 100 dernières ventes

df_last_100 = df.sort_values("Transaction_Date", ascending=False).head(100)

df_last_100["Transaction_Date"] = pd.to_datetime(df_last_100["Transaction_Date"]).dt.strftime("%Y-%m-%d")

df_last_100 = df_last_100[
    ["Transaction_Date", "Gender", "Location", "Product_Category",
     "Quantity", "Avg_Price", "Discount_pct"]]

In [ ]:
import pandas as pd
import calendar as cal
import plotly.express as px
import plotly.graph_objects as go

import dash
from dash import html, dcc, dash_table, Input, Output
import dash_bootstrap_components as dbc


# PREPARATION DONNEES --------------------------
df = pd.read_csv("data.csv")

cols = [
    "CustomerID",
    "Gender",
    "Location",
    "Product_Category",
    "Quantity",
    "Avg_Price",
    "Transaction_Date",
    "Month",
    "Discount_pct"
]

df = df[cols].copy()
df["CustomerID"] = df["CustomerID"].fillna(0).astype(int)
df["Transaction_Date"] = pd.to_datetime(df["Transaction_Date"])
df["Total_price"] = df["Quantity"] * df["Avg_Price"] * (1 - df["Discount_pct"] / 100)

CURRENT_MONTH = 12


# FONCTIONS ----------------------------
def meilleure_vente(data, top=10, ascending=False):
    freq = (
        data.groupby("Product_Category")["Quantity"]
        .sum()
        .sort_values(ascending=ascending)
    )
    return freq.head(top)


def indicateur_du_mois(data, current_month=12, freq=True, abbr=False):
    data_mois = data[data["Month"] == current_month]

    if freq:
        result = data_mois["Quantity"].sum()
    else:
        result = data_mois["Total_price"].sum()

    month_name = cal.month_abbr[current_month] if abbr else cal.month_name[current_month]
    return month_name, result


def barplot_top_10_ventes(data, current_month=12):
    data_mois = data[data["Month"] == current_month].copy()

    top = meilleure_vente(data_mois, top=10, ascending=False).reset_index()

    data_top = data_mois[data_mois["Product_Category"].isin(top["Product_Category"])]

    ventes = (
        data_top.groupby(["Product_Category", "Gender"])["Quantity"]
        .sum()
        .reset_index()
    )

    ventes["Product_Category"] = pd.Categorical(
        ventes["Product_Category"],
        categories=top["Product_Category"].tolist(),
        ordered=True
    )
    ventes = ventes.sort_values("Product_Category", ascending=True)

    fig = px.bar(
        ventes,
        x="Quantity",
        y="Product_Category",
        color="Gender",
        barmode="group",
        title="Frequence des 10 meilleures ventes",
        labels={
            "Product_Category": "Catégorie du produit",
            "Quantity": "Total vente",
            "Gender": "Sexe"
        },
        color_discrete_map={
            "F": "#636EFA",
            "M": "#EF553B"
        },
        category_orders={
            "Product_Category": top["Product_Category"].tolist(),
            "Gender": ["F", "M"]
        }
    )

    fig.update_layout(
        margin=dict(l=10, r=10, t=45, b=10),
        paper_bgcolor="#E5ECF6",
        plot_bgcolor="#E5ECF6",
        legend_title_text="Sexe",
        title_font=dict(size=14),
        font=dict(size=11)
    )

    return fig


def plot_evolution_chiffre_affaire(data):
    data_copy = data.copy().set_index("Transaction_Date")
    evolution = data_copy["Total_price"].resample("W").sum().reset_index()

    fig = px.line(
        evolution,
        x="Transaction_Date",
        y="Total_price",
        title="Evolution du chiffre d'affaire par semaine",
        labels={
            "Transaction_Date": "Semaine",
            "Total_price": "Chiffre d'affaire"
        }
    )

    fig.update_traces(line=dict(color="#636EFA", width=2))

    fig.update_layout(
        margin=dict(l=10, r=10, t=45, b=10),
        paper_bgcolor="#E5ECF6",
        plot_bgcolor="#E5ECF6",
        title_font=dict(size=14),
        font=dict(size=11)
    )
    return fig


def plot_chiffre_affaire_mois(data, current_month):
    previous_month = 12 if current_month == 1 else current_month - 1

    mois_courant = indicateur_du_mois(
        data, current_month=current_month, freq=False, abbr=False
    )
    mois_precedent = indicateur_du_mois(
        data, current_month=previous_month, freq=False, abbr=False
    )

    fig = go.Figure()

    fig.add_trace(
        go.Indicator(
            value=mois_courant[1],
            mode="number+delta",
            number={"valueformat": ".3s"},
            delta={
                "reference": mois_precedent[1],
                "valueformat": ".3s"
            },
            title={"text": f"{mois_courant[0]}"}
        )
    )

    fig.update_layout(
        margin=dict(l=10, r=10, t=25, b=0),
        paper_bgcolor="white",
        plot_bgcolor="white",
        font=dict(color="#2a3f5f")
    )

    return fig


def plot_vente_mois(data, current_month, abbr=False):
    previous_month = 12 if current_month == 1 else current_month - 1

    mois_courant = indicateur_du_mois(
        data, current_month=current_month, freq=True, abbr=abbr
    )
    mois_precedent = indicateur_du_mois(
        data, current_month=previous_month, freq=True, abbr=abbr
    )

    fig = go.Figure()

    fig.add_trace(
        go.Indicator(
            mode="number+delta",
            value=mois_courant[1],
            delta={"reference": mois_precedent[1]},
            title={"text": f"{mois_courant[0]}"}
        )
    )

    fig.update_layout(
        margin=dict(l=10, r=10, t=25, b=0),
        paper_bgcolor="white",
        plot_bgcolor="white",
        font=dict(color="#2a3f5f")
    )

    return fig


def table_last_100(data, current_month):
    df_last_100 = (
        data[data["Month"] == current_month]
        .sort_values("Transaction_Date", ascending=False)
        .head(100)
        .copy()
    )

    df_last_100["Date"] = df_last_100["Transaction_Date"].dt.strftime("%Y-%m-%d")

    df_last_100 = df_last_100[
        [
            "Date",
            "Gender",
            "Location",
            "Product_Category",
            "Quantity",
            "Avg_Price",
            "Discount_pct"
        ]
    ]

    return df_last_100



# TABLEAU DE BORD ------------------------------------------------

app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

zone_options = [
    {"label": z, "value": z}
    for z in sorted(df["Location"].dropna().unique())
]

app.layout = dbc.Container(
    [
        dbc.Row(
            [
                dbc.Col(
                    html.Div(
                        html.H1(
                            "ECAP Store",
                            style={
                                "margin": "0",
                                "fontSize": "24px",
                                "fontWeight": "700",
                                "color": "#1f2d3d"
                            }
                        ),
                        style={
                            "height": "100%",
                            "display": "flex",
                            "alignItems": "center",
                            "paddingLeft": "12px",
                            "backgroundColor": "#c7e6ef"
                        }
                    ),
                    md=6
                ),
                dbc.Col(
                    html.Div(
                        dcc.Dropdown(
                            id="zone-dropdown",
                            options=zone_options,
                            value=None,
                            clearable=True,
                            placeholder="Choisissez des zones",
                            style={"width": "330px"}
                        ),
                        style={
                            "height": "100%",
                            "display": "flex",
                            "alignItems": "center",
                            "justifyContent": "center",
                            "backgroundColor": "#c7e6ef"
                        }
                    ),
                    md=6
                ),
            ],
            className="g-0",
            style={"height": "7vh"}
        ),
# INDICATEURS 
        dbc.Row(
            [
                dbc.Col(
                    [
                        dbc.Row(
                            [
                                dbc.Col(
                                    dcc.Graph(
                                        id="ca-mois",
                                        config={"displayModeBar": False},
                                        style={"height": "100%"}
                                    ),
                                    md=6,
                                    style={"height": "100%", "padding": "10px"}
                                ),
                                dbc.Col(
                                    dcc.Graph(
                                        id="vente-mois",
                                        config={"displayModeBar": False},
                                        style={"height": "100%"}
                                    ),
                                    md=6,
                                    style={"height": "100%", "padding": "10px"}
                                ),
                            ],
                            className="g-0",
                            style={"height": "25vh"}
                        ),
# BARPLOT TOP 10 VENTES
                        dbc.Row(
                            [
                                dbc.Col(
                                    dcc.Graph(
                                        id="top-10-ventes",
                                        config={"displayModeBar": False},
                                        style={"height": "100%"}
                                    ),
                                    md=12,
                                    style={"height": "100%", "padding": "10px"}
                                ),
                            ],
                            className="g-0",
                            style={"height": "68vh"}
                        ),
                    ],
                    md=5
                ),
# GRAPH EVOLUTION CA
                dbc.Col(
                    [
                        dbc.Row(
                            [
                                dbc.Col(
                                    dcc.Graph(
                                        id="evolution-ca",
                                        config={"displayModeBar": False},
                                        style={"height": "100%"}
                                    ),
                                    md=12,
                                    style={"height": "100%", "padding": "10px"}
                                ),
                            ],
                            className="g-0",
                            style={"height": "48vh"}
                        ),
# TABLE 100 VENTES
                        dbc.Row(
                            [
                                dbc.Col(
                                    html.Div(
                                        [
                                            html.H4(
                                                "Table des 100 dernières ventes",
                                                style={
                                                    "margin": "0 0 8px 0",
                                                    "fontSize": "18px",
                                                    "fontWeight": "600"
                                                }
                                            ),
                                            html.Div(
                                                dash_table.DataTable(
                                                    id="table-last-100",
                                                    page_size=10,
                                                    page_action="native",
                                                    sort_action="native",
                                                    filter_action="native",
                                                    style_table={
                                                        "width": "100%",
                                                        "overflowX": "auto",
                                                        "overflowY": "auto",
                                                        "maxHeight": "34vh"
                                                    },
                                                    style_cell={
                                                        "textAlign": "center",
                                                        "padding": "4px",
                                                        "fontFamily": "Arial",
                                                        "fontSize": "12px",
                                                        "minWidth": "88px",
                                                        "width": "88px",
                                                        "maxWidth": "140px",
                                                        "whiteSpace": "nowrap",
                                                        "border": "1px solid #D6D6D6"
                                                    },
                                                    style_header={
                                                        "fontWeight": "bold",
                                                        "backgroundColor": "#F6F6F6",
                                                        "border": "1px solid #D6D6D6",
                                                        "fontSize": "12px"
                                                    },
                                                    style_data={
                                                        "backgroundColor": "white",
                                                        "border": "1px solid #E1E1E1"
                                                    },
                                                    css=[
                                                        {
                                                            "selector": ".dash-spreadsheet-menu",
                                                            "rule": "display: none;"
                                                        }
                                                    ]
                                                ),
                                                style={
                                                    "flex": "1",
                                                    "overflow": "auto"
                                                }
                                            )
                                        ],
                                    ),
                                    md=12,
                                    style={"height": "100%"}
                                ),
                            ],
                            className="g-0",
                            style={"height": "45vh"}
                        ),
                    ],
                    md=7
                ),
            ],
            className="g-0",
            style={"height": "93vh"}
        ),
    ],
    fluid=True,
    className="p-0",
    style={
        "height": "100vh",
        "backgroundColor": "white",
        "overflow": "hidden"
    }
)


# CALLBACK ---------------------------
@app.callback(
    Output("ca-mois", "figure"),
    Output("vente-mois", "figure"),
    Output("top-10-ventes", "figure"),
    Output("evolution-ca", "figure"),
    Output("table-last-100", "data"),
    Output("table-last-100", "columns"),
    Input("zone-dropdown", "value")
)
def update_dashboard(selected_zone):
    data_filtered = df.copy()

    if selected_zone:
        data_filtered = data_filtered[data_filtered["Location"] == selected_zone]

    fig_ca = plot_chiffre_affaire_mois(data_filtered, CURRENT_MONTH)
    fig_ventes = plot_vente_mois(data_filtered, CURRENT_MONTH, False)
    fig_top = barplot_top_10_ventes(data_filtered, CURRENT_MONTH)
    fig_evolution = plot_evolution_chiffre_affaire(data_filtered)

    table_df = table_last_100(data_filtered, CURRENT_MONTH)

    return (
        fig_ca,
        fig_ventes,
        fig_top,
        fig_evolution,
        table_df.to_dict("records"),
        [{"name": c.replace("_", " "), "id": c} for c in table_df.columns]
    )


if __name__ == "__main__":
    app.run(debug=False, port=8055, jupyter_mode="external")

Dash app running on http://127.0.0.1:8055/
